# UAVDT vehicle detection: YOLO and RF-DETR training

Runs the whole pipeline: download UAVDT → convert to YOLO and COCO → train YOLO → train RF-DETR → compare.

**Before running**

1. Push your latest code to GitHub. The notebook clones the repo fresh each session, so whatever is on `REPO_BRANCH` is what trains. Nothing needs uploading to Drive by hand.
2. If the repo is private, create a GitHub personal access token with read access to it, then in Colab open the key icon (**Secrets**) in the left sidebar, add it as `GITHUB_TOKEN`, and turn on notebook access. Public repos need no token.
3. Open this notebook in Colab (**File → Open notebook → GitHub** works too), then **Runtime → Change runtime type → GPU** (A100 or L4 if you have them; T4 works but is much slower).
4. Set `REPO_URL`, `REPO_BRANCH`, and `DRIVE_DIR` in the settings cell below.
5. Set `QUICK_TEST = True` for a first run. It uses a few hundred images and 1 epoch, so the whole notebook finishes in minutes and any setup problem shows up before you spend hours of GPU time. Then set it back to `False` and **Runtime → Run all**.

**Where things go**

- Code: cloned to `/content/group_project` on the Colab machine. Rerunning the setup cell pulls the latest commit on `REPO_BRANCH` and discards local edits there, so make changes in the repo, not on Colab.
- Dataset: `/content/data` on the Colab machine. Local disk is much faster than Drive for 40k small images, but it is wiped when the session ends, so each new session downloads again (13 GB, usually 5–15 min).
- Trained models, scores, and graphs: `DRIVE_DIR/models/yolo/<run time>/` and `DRIVE_DIR/models/rf-detr/<run time>/` on Drive, so they survive the session ending. Checkpoints are written there every epoch. Each run folder also gets `code_version.txt` with the commit it was trained from.


In [ ]:
# ---- Settings ----------------------------------------------------------------
# GitHub repo to clone the code from, and the branch to train.
REPO_URL = "https://github.com/JoelWeberDev/CP-467_group_project.git"
REPO_BRANCH = "main"

# Folder on Google Drive where trained models are saved (created if missing).
DRIVE_DIR = "/content/drive/MyDrive/CP-467/group_project"

# Run the whole pipeline on a tiny slice (every 200th frame, 1 epoch) to check it works.
QUICK_TEST = False

# Keep every Nth video frame. Neighbouring frames are nearly identical, so 2-5
# cuts epoch time roughly by that factor at little cost in accuracy.
FRAME_STRIDE = 1

TRAIN_YOLO = True
YOLO_WEIGHTS = "yolo26l.pt"  # Pretrained weights; downloaded automatically.
YOLO_EPOCHS = 50
YOLO_IMGSZ = 640  # Images are 1024x540. 1024 helps with the many small vehicles but is ~2.5x slower.
YOLO_BATCH = 16  # Lower this if you get CUDA out-of-memory errors.

TRAIN_RFDETR = True
RFDETR_EPOCHS = 20

# ---- Derived paths (no need to edit) -------------------------------------------
import os

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/group_project"
    DATA_DIR = "/content/data"
else:
    # Running locally from the project root: use this checkout and its own folders.
    REPO_DIR = os.path.abspath(".")
    DRIVE_DIR = REPO_DIR
    DATA_DIR = os.path.join(REPO_DIR, "data")

CODE_DIR = os.path.join(REPO_DIR, "code")
if QUICK_TEST:
    FRAME_STRIDE, YOLO_EPOCHS, RFDETR_EPOCHS = 200, 1, 1
    PREP_DIR = os.path.join(DATA_DIR, "quick_test")
    MODELS_DIR = os.path.join(DRIVE_DIR, "models", "quick_test")
else:
    PREP_DIR = DATA_DIR
    MODELS_DIR = os.path.join(DRIVE_DIR, "models")

## 1. Environment

In [ ]:
import subprocess
import sys


def git(*args, cwd=None):
    """Run git and return its output. Raises with git's own message on failure."""
    result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr
        if token:
            message = message.replace(token, "***")  # Never echo the token back.
        raise RuntimeError(f"git {args[0]} failed:\n{message}")
    return result.stdout.strip()


token = None
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    assert "OWNER/REPO" not in REPO_URL, "Set REPO_URL in the settings cell to the team's GitHub repo."
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None  # No secret set: fine for a public repo.

    clone_url = REPO_URL
    if token:
        clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)

    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        # Pull the latest commit; local edits on Colab are discarded on purpose.
        git("remote", "set-url", "origin", clone_url, cwd=REPO_DIR)
        git("fetch", "--depth", "1", "origin", REPO_BRANCH, cwd=REPO_DIR)
        git("checkout", "-B", REPO_BRANCH, "FETCH_HEAD", cwd=REPO_DIR)
        git("reset", "--hard", "FETCH_HEAD", cwd=REPO_DIR)
    else:
        git("clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, REPO_DIR)
    # Drop the token from the saved remote so it is not left in .git/config.
    git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)

assert os.path.isfile(os.path.join(CODE_DIR, "dataset_prep.py")), (
    f"Project code not found at {CODE_DIR}. Check REPO_URL and REPO_BRANCH in the settings cell."
)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

try:
    CODE_COMMIT = git("rev-parse", "HEAD", cwd=REPO_DIR)
    CODE_COMMIT_INFO = git("log", "-1", "--format=%h %s (%an, %ad)", "--date=short", cwd=REPO_DIR)
except RuntimeError:
    CODE_COMMIT, CODE_COMMIT_INFO = "unknown", "unknown (not a git checkout)"


def record_code_version(run_dir: str) -> None:
    """Note which commit a run was trained from, next to its results."""
    with open(os.path.join(run_dir, "code_version.txt"), "w", encoding="utf-8") as handle:
        handle.write(f"repo: {REPO_URL if IN_COLAB else REPO_DIR}\nbranch: {REPO_BRANCH}\ncommit: {CODE_COMMIT}\n{CODE_COMMIT_INFO}\n")


for path in (DATA_DIR, PREP_DIR, os.path.join(MODELS_DIR, "yolo"), os.path.join(MODELS_DIR, "rf-detr")):
    os.makedirs(path, exist_ok=True)

print("Code:   ", REPO_DIR, "|", CODE_COMMIT_INFO)
print("Data:   ", DATA_DIR)
print("Models: ", MODELS_DIR)
print("Quick test" if QUICK_TEST else "Full run", f"| frame stride {FRAME_STRIDE}")


In [ ]:
# Install the training libraries (pinned to the versions the code was written against).
# dataset_tools is deliberately not installed: it pins exact numpy/pandas/pillow
# versions that break Colab's torch. The download below does not need it.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.216", "rfdetr[train]==1.12.1"],
        check=True,
    )

import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, {gpu.total_memory / 1024**3:.0f} GB")
else:
    print("WARNING: no GPU. In Colab use Runtime -> Change runtime type -> GPU. Training on CPU is only usable for QUICK_TEST.")

## 2. Download UAVDT

In [ ]:
from download_data import downloadUAVDTDirect

source_dir = downloadUAVDTDirect(DATA_DIR)

In [ ]:
# A few annotated frames as a sanity check (same drawing as preview_dataset.py).
import json
import matplotlib.pyplot as plt
from preview_dataset import _annotated_image, _class_colors

colors = _class_colors(os.path.join(source_dir, "meta.json"))
samples = [("train", "M0101_img000001.jpg"), ("train", "M0204_img000001.jpg"),
           ("test", "M0203_img000500.jpg"), ("test", "M1401_img000300.jpg")]
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for axis, (split, name) in zip(axes.flat, samples):
    if os.path.isfile(os.path.join(source_dir, split, "img", name)):
        image = _annotated_image(source_dir, split, name, colors)
        axis.imshow(image[:, :, ::-1])
    axis.set_title(f"{split}/{name}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Convert to YOLO and COCO

Keeps the 3 classes car, truck, and bus, takes 20% of the train sequences as validation, and skips the sequences listed in `EXCLUDED_SEQUENCES` in `dataset_prep.py`. Takes under a minute: images are hard linked, not copied.

In [ ]:
from dataset_prep import prepareUAVDT

# replace=True so changes to FRAME_STRIDE or QUICK_TEST always take effect.
prepared = prepareUAVDT(source_dir, PREP_DIR, frame_stride=FRAME_STRIDE, replace=True)
YOLO_DATASET = prepared["yolo"]
COCO_DATASET = prepared["coco"]

## 4. Train YOLO

Ultralytics prints a progress bar per epoch. Results land in `models/yolo/<run time>/`: `weights/best.pt`, `epoch_scores.csv`, `performance.png`, and `run_summary.json`.

In [ ]:
from IPython.display import Image, display


def show_run(output_dir: str) -> dict:
    with open(os.path.join(output_dir, "run_summary.json"), encoding="utf-8") as handle:
        summary = json.load(handle)
    print(json.dumps({key: summary[key] for key in ("best_epoch_by_val_f1", "fps", "epoch_count")}, indent=2))
    graph = os.path.join(output_dir, "performance.png")
    if os.path.isfile(graph):
        display(Image(filename=graph, width=700))
    return summary


yolo_run_dir = None
if TRAIN_YOLO:
    from train_yolo import trainYOLO

    yolo_run_dir = trainYOLO(
        YOLO_DATASET,
        os.path.join(MODELS_DIR, "yolo"),
        epochs=YOLO_EPOCHS,
        model_weights=YOLO_WEIGHTS,
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
    )
    record_code_version(yolo_run_dir)
    print("YOLO run saved to:", yolo_run_dir)
    yolo_summary = show_run(yolo_run_dir)

## 5. Train RF-DETR

Results land in `models/rf-detr/<run time>/`: `checkpoint_best_total.pth`, `epoch_scores.csv`, `performance.png`, and `run_summary.json`.

In [ ]:
import gc
import importlib

# Free YOLO's GPU memory before RF-DETR starts.
gc.collect()
torch.cuda.empty_cache()

rfdetr_run_dir = None
if TRAIN_RFDETR:
    # The file name has a hyphen, so it cannot be imported with a plain import statement.
    trainRFDETR = importlib.import_module("train_rf-detr").trainRFDETR

    rfdetr_run_dir = trainRFDETR(COCO_DATASET, os.path.join(MODELS_DIR, "rf-detr"), epochs=RFDETR_EPOCHS)
    record_code_version(rfdetr_run_dir)
    print("RF-DETR run saved to:", rfdetr_run_dir)
    rfdetr_summary = show_run(rfdetr_run_dir)

## 6. Compare

Each model's best epoch by validation F1. Validation is the held-out train sequences, not the test split.

In [ ]:
import pandas as pd

rows = {}
for label, run_dir in (("YOLO", yolo_run_dir), ("RF-DETR", rfdetr_run_dir)):
    if not run_dir:
        continue
    with open(os.path.join(run_dir, "run_summary.json"), encoding="utf-8") as handle:
        summary = json.load(handle)
    best = summary.get("best_epoch_by_val_f1") or {}
    scores = best.get("validation", {})
    rows[label] = {
        "best epoch": best.get("epoch"),
        "precision": scores.get("precision"),
        "recall": scores.get("recall"),
        "F1": scores.get("f1"),
        "mAP@0.5": scores.get("map_50"),
        "mAP@0.5:0.95": scores.get("map_50_95"),
        "images/sec": (summary.get("fps") or {}).get("images_per_second"),
        "run folder": os.path.relpath(run_dir, DRIVE_DIR),
    }

pd.DataFrame(rows).T.round(3) if rows else print("No runs were trained.")

## Notes

- **Session limits.** Colab can disconnect partway through a long run. Checkpoints are already on Drive, but `trainYOLO` and `trainRFDETR` start fresh runs; they do not resume. If that becomes a problem, lower the epochs, raise `FRAME_STRIDE`, or set `TRAIN_YOLO = False` and train the two models in separate sessions.
- **Changing code.** Edit locally, commit, push, then rerun the setup cell (section 1) to pull it. Python caches imported modules, so after pulling changes to `code/` use **Runtime → Restart session** and run all again.
- **Test split.** Training only scores the validation split. The test split (`images/test`, `test/_annotations.coco.json`) is prepared but not yet evaluated.